In [ ]:
import os 

os.chdir("..")

In [ ]:
from jp_imports import JPTrade
from jp_qcew import CleanQCEW
import polars as pl
import pymc as pm
import pandas as pd
from prprod_repl import ProdRepl
import arviz_base as azb
import arviz_plots as azp
azp.style.use("arviz-variat")


jt = JPTrade()
pr = ProdRepl()
cq = CleanQCEW(saving_dir="data/")

In [ ]:
data = pr.reg_data()
data = data.sort(by=["year", "month", "sector_code", "name"]).to_pandas()
data

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd

# 1. Create a continuous time index (Year + Month)
# This creates a sorted list of unique year-month combinations
time_coords = data[['year', 'month']].drop_duplicates().sort_values(['year', 'month'])
time_coords['time_idx'] = range(len(time_coords))

# Merge this index back into the main dataframe
data = data.merge(time_coords, on=['year', 'month'], how='left')

# 2. Preprocessing categorical indexes
data["sector_idx"], sectors = pd.factorize(data["sector_code"])
data["muni_idx"], munis = pd.factorize(data["name"])

n_sectors = len(sectors)
n_periods = len(time_coords) # Total number of months in the timeseries

# Coordinate system for the model
coords = {
    "time": range(n_periods),
    "sector": sectors,
    "muni": munis,
    "fourier_features": ["sin", "cos"]
}

# 3. Seasonal features (using the integer month column)
# Scaling month (1-12) to a circle (0 to 2*pi)
month_scaled = 2 * np.pi * data["month"].values / 12
sin_t = np.sin(month_scaled)
cos_t = np.cos(month_scaled)

with pm.Model(coords=coords) as import_model:
    # --- 1. Sector-Level Random Walk ---
    # Global drift across all sectors
    state_drift_mu = pm.Normal("state_drift_mu", mu=0, sigma=0.01)
    
    # Specific drift for each sector (how much they grow/shrink monthly)
    sector_drift = pm.Normal("sector_drift", mu=state_drift_mu, sigma=0.05, dims="sector")
    
    # Monthly volatility
    sigma_rw = pm.Exponential("sigma_rw", 2.0)
    
    # Innovation offsets (Non-centered)
    z_innovations = pm.Normal("z_innovations", 0, 1, dims=("sector", "time"))
    
    # Initial log-import value per sector
    # We use the mean of the first observed month for each sector as a prior
    init_val = np.log(data.groupby("sector_idx")["imports"].first().values + 1)
    init_import = pm.Normal("init_import", mu=init_val, sigma=0.5, dims="sector")
    
    # The latent random walk path
    # Shape: (n_sectors, n_periods)
    sector_path = pm.Deterministic(
        "sector_path",
        init_import[:, None] + (sector_drift[:, None] + z_innovations * sigma_rw).cumsum(axis=1),
        dims=("sector", "time")
    )

    # --- 2. Fourier Seasonality ---
    # Captures the repeating annual cycle per sector
    beta_fourier = pm.Normal("beta_fourier", mu=0, sigma=0.5, dims=("sector", "fourier_features"))
    
    # Extract the sin/cos components for each row based on sector
    seasonality = (
        beta_fourier[data.sector_idx.values, 0] * sin_t + 
        beta_fourier[data.sector_idx.values, 1] * cos_t
    )

    # --- 3. Employment Relationship ---
    # Log-log relationship: Imports as a function of local employment
    beta_employment = pm.HalfNormal("beta_employment", sigma=1.0)
    log_emp = np.log(data["employment"].values + 1)

    # --- 4. Expected Value ---
    # Match the sector/time path to the long-form data rows
    mu = pm.Deterministic(
        "mu",
        sector_path[data.sector_idx.values, data.time_idx.values] + 
        (beta_employment * log_emp) + 
        seasonality
    )

    # --- 5. Likelihood ---
    sigma_obs = pm.Exponential("sigma_obs", 1.0)
    
    pm.StudentT(
        "obs",
        nu=4,
        mu=mu,
        sigma=sigma_obs,
        observed=np.log(data["imports"].values + 1)
    )

    # --- Inference ---
    trace = pm.sample(1000, tune=1000, target_accept=0.99)

In [ ]:
trace

In [ ]:
import altair as alt
import arviz as az
import pandas as pd
import numpy as np

def plot_import_forecast(sector_code: str, muni_name: str, trace):
    # 1. Data Prep & Indexing
    df = data.copy()
    
    # Filter for specific muni and sector
    subset = df[(df["sector_code"] == sector_code) & (df["name"] == muni_name)].copy()
    subset = subset.sort_values(["year", "month"])
    
    if subset.empty:
        print(f"No data found for Sector {sector_code} in {muni_name}")
        return

    # Create datetime objects for plotting
    subset["date"] = pd.to_datetime(subset[["year", "month"]].assign(day=1))
    obs_dates = subset["date"].values
    
    # 2. Extract Posterior for 'mu' (Expected log imports)
    # We select by the indices corresponding to our subset
    # 'mu' shape is (chain, draw, total_observations)
    # We need the indices of the rows in the original 'data' that match our subset
    row_indices = subset.index.values
    
    # Get posterior samples for these specific rows
    post_mu = trace.posterior["mu"].values # (chains, draws, N)
    post_mu_flat = post_mu.reshape(-1, post_mu.shape[-1])[:, row_indices]
    
    # Convert from log space back to levels (exp)
    import_samples = np.exp(post_mu_flat) - 1 

    # 3. Calculate Stats
    median_imports = np.median(import_samples, axis=0)
    hdi_95 = az.hdi(import_samples, hdi_prob=0.95)
    hdi_50 = az.hdi(import_samples, hdi_prob=0.50)

    # 4. Create Plotting DataFrame
    plot_df = pd.DataFrame({
        "date": obs_dates,
        "median": median_imports,
        "hdi_95_low": hdi_95[:, 0],
        "hdi_95_high": hdi_95[:, 1],
        "hdi_50_low": hdi_50[:, 0],
        "hdi_50_high": hdi_50[:, 1],
        "observed": subset["imports"].values
    })

    # 5. Build Altair Chart
    base = alt.Chart(plot_df).encode(
        x=alt.X("date:T", title="Date")
    )

    # Confidence Intervals (95% and 50%)
    band_95 = base.mark_area(opacity=0.15, color="steelblue").encode(
        y="hdi_95_low:Q",
        y2="hdi_95_high:Q"
    )

    band_50 = base.mark_area(opacity=0.3, color="steelblue").encode(
        y="hdi_50_low:Q",
        y2="hdi_50_high:Q"
    )

    # Median Line
    line = base.mark_line(strokeWidth=2, color="steelblue").encode(
        y=alt.Y("median:Q", title="Imports ($)", scale=alt.Scale(zero=False))
    )

    # Observed Points (Actual Data)
    points = base.mark_point(size=50, filled=True, color="black").encode(
        y="observed:Q",
        tooltip=["date", "observed"]
    )

    # Combine
    chart = (
        alt.layer(band_95, band_50, line, points)
        .properties(
            title={
                "text": f"Import Model: {muni_name}",
                "subtitle": f"Sector: {sector_code} | Includes Seasonality & Employment Proxy",
                "anchor": "start"
            },
            width=800,
            height=400
        )
        .configure_axis(gridDash=[3, 3], gridColor="#eeeeee")
    )

    return chart

In [ ]:
data

In [ ]:
plot_import_forecast("11", "adjuntas", trace)